# Ames Mutagenicity: Classical and Graph Neural Network Models

## Part 18 — One decision rule for every comparison

The manuscript argues that this literature applies a looser standard where a looser standard
favours the model. Its own tables do not yet meet the standard they ask for. The threshold-free
comparison, which the paper calls primary, reports intervals and no $p$-values and carries no
correction. The lift comparison, on which no architecture exceeds the baseline, carries Holm.
That asymmetry runs in the direction the paper criticises.

This notebook computes bootstrap $p$-values for the comparisons that can carry them, states a
correction family for each, and writes one table giving the family, the correction and the
threshold. Where a family is left uncorrected the reason is recorded rather than assumed.
Nothing is trained.

**Contents**

1. Environment, data and stored quantities
2. Threshold-free comparisons
3. The specificity check
4. The retained-fraction sweep
5. One decision rule, tabulated
6. Summary

One item from the manuscript review is deliberately absent. The combination of the three
within-architecture correlations, which assumed an independence the design does not have, was
removed from the manuscript rather than replaced with a dependence-aware estimate; the three
intervals are reported and each contains zero, which is what the data support.

*Approximate runtime: about twenty-five minutes on CPU. The specificity check dominates it,
recomputing signed attributions for thirty models on two evaluation sets; the sweep is a few
minutes because the tie-breaking is vectorised over draws. Each fraction of the sweep is
cached on its own, so a dropped session costs one fraction rather than the stage.*

## Pre-registered readings

Fixed before the notebook runs.

### The correction families

Four comparisons are made against the degree ranking, and they do not form one family. The
grouping below is a choice, and other groupings are defensible; what matters is that it is
declared before the numbers are seen.

| family | members | why these |
|---|---|---|
| **Attribution, threshold-free** (5 tests) | GIN, GAT, GCN under Integrated Gradients; Gradient $\times$ Input; Saliency | all ask whether a trained attribution beats the baseline on the same measure and the same molecules |
| **Specificity** (3 tests) | the difference of differences for each architecture | three pre-specified architectures, one question |
| **Retained-fraction sweep** (12 tests) | three architectures at the four larger fractions | reported uncorrected, for the reason given below |
| **Sweep, smallest fraction** (3 tests) | GIN, GAT, GCN at $k = 0.10$ | a claim of presence, so corrected |
| **Structural ranking panel** (33 cells) | eleven rankings in three regimes | a screen that selected the baseline, not a test |

The random ranking is compared with the degree ranking as a check on the measure, not as a
claim about a model, and sits outside every family. Its interval and $p$ are reported
uncorrected and labelled as a control.

### The readings

| result | reading |
|---|---|
| GIN's threshold-free advantage survives Holm across the five tests of its family | the primary comparison stands as reported and the manuscript states the correction |
| it does not survive | the claim is reported as uncorrected; the direction and interval stand, the significance claim does not |
| Saliency survives and GIN does not, or the reverse | both are reported with their corrected values and no ordering is claimed between them |
| the specificity difference of differences survives Holm across three | the Conclusions sentence about GIN stands |
| it does not survive | that sentence is qualified and the result reported as uncorrected |
| GIN or GAT survives Holm at $k = 0.10$ | the statement that trained models lead at the top of the ranking stands as a corrected result |
| neither survives | that statement is reported as uncorrected and the interval carries it |
| the sweep recomputed here agrees with the published table at $k = 0.30$ within $0.01$ | the published table stands and the new $p$-values are appended to it |
| it does not agree | the recomputed table replaces the published one, and the difference is reported as an effect of the tie-break budget rather than left as two values for one quantity |

### On corrections and the direction of a claim

A correction can only weaken a claim that something is present. It cannot weaken a claim that
nothing is present: the sweep is used to say that no architecture exceeds the baseline at the
reference fraction, and correcting the twelve cells that carry that claim would only make it
harder to overturn.
Leaving the sweep uncorrected is therefore the conservative choice for the claim drawn from
it, which is why it is left uncorrected and why that is stated rather than assumed. For the
positive claims of Sections 2 and 3 the opposite holds, and there the correction is applied.

## 1. Environment, data and stored quantities

In [1]:
!pip install -q rdkit
!pip install -q torch_geometric

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 37.4/37.4 MB 32.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.4/64.4 kB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 40.1 MB/s eta 0:00:00


In [2]:
from google.colab import drive
drive.mount('/content/drive')
import os
WORK = "/content/drive/MyDrive/ames_pipeline"
DATADIR, R, EXPL, MODELS = f"{WORK}/data", f"{WORK}/results", f"{WORK}/explain", f"{WORK}/models"
os.makedirs(R, exist_ok=True)
print("results:", R)

Mounted at /content/drive
results: /content/drive/MyDrive/ames_pipeline/results


In [3]:
%%writefile /content/ames9.py
"""Definitions shared across Part 9, matching the protocol used in Parts 2 to 5."""
import random, warnings
import numpy as np, torch, torch.nn as nn, torch.nn.functional as F
from collections import defaultdict
from rdkit import Chem, RDLogger, DataStructs
from rdkit.Chem.Scaffolds import MurckoScaffold
from rdkit.Chem import rdFingerprintGenerator as rfg
from torch_geometric.data import Data
from torch_geometric.loader import DataLoader
from torch_geometric.nn import (GCNConv, GINConv, GINEConv, GATConv,
                                global_mean_pool, global_add_pool)
from sklearn.metrics import roc_auc_score

RDLogger.DisableLog("rdApp.*"); warnings.filterwarnings("ignore")
FP_RADIUS, FP_BITS = 2, 2048
ATOM_DIM, BOND_DIM = 24, 6
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
ATOM_LIST     = ['C','N','O','S','F','Cl','Br','I','P','B','Si','Se','other']
HYBRIDIZATION = ['SP','SP2','SP3','SP3D','SP3D2','other']
morgan_gen = rfg.GetMorganGenerator(radius=FP_RADIUS, fpSize=FP_BITS)

def set_seed(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(s)

def safe_scaffold(smi):
    mol = Chem.MolFromSmiles(smi)
    if mol is None: return ""
    try: return MurckoScaffold.MurckoScaffoldSmiles(mol=mol)
    except Exception:
        try: return Chem.MolToSmiles(MurckoScaffold.GetScaffoldForMol(mol))
        except Exception: return smi

def make_scaffold_split(d, seed, frac=(0.8, 0.1, 0.1)):
    groups = defaultdict(list)
    for i, smi in zip(d.index, d["canonical"].values):
        groups[safe_scaffold(smi)].append(i)
    gs = list(groups.values()); rng = np.random.default_rng(seed)
    gs.sort(key=lambda g: (-len(g), rng.random()))
    n = len(d); a, b = int(frac[0]*n), int((frac[0]+frac[1])*n)
    tr, va, te = [], [], []
    for g in gs:
        if len(tr)+len(g) <= a: tr += g
        elif len(tr)+len(va)+len(g) <= b: va += g
        else: te += g
    return d.loc[tr], d.loc[va], d.loc[te]

def fp_array(smi):
    mol = Chem.MolFromSmiles(smi)
    if mol is None: return None
    arr = np.zeros((FP_BITS,), dtype=np.int8)
    DataStructs.ConvertToNumpyArray(morgan_gen.GetFingerprint(mol), arr)
    return arr

def build_X(d):
    X, keep = [], []
    for i, row in d.iterrows():
        a = fp_array(row["canonical"])
        if a is not None: X.append(a); keep.append(i)
    return np.array(X), d.loc[keep].reset_index(drop=True)

def one_hot(x, allowable):
    if x not in allowable: x = allowable[-1]
    return [int(x == a) for a in allowable]

def atom_features(a):
    f  = one_hot(a.GetSymbol(), ATOM_LIST) + one_hot(str(a.GetHybridization()), HYBRIDIZATION)
    f += [a.GetDegree(), a.GetFormalCharge(), int(a.GetIsAromatic()),
          a.GetTotalNumHs(), int(a.IsInRing())]
    return f

def bond_features(b):
    bt = b.GetBondType()
    return [int(bt == Chem.BondType.SINGLE), int(bt == Chem.BondType.DOUBLE),
            int(bt == Chem.BondType.TRIPLE), int(bt == Chem.BondType.AROMATIC),
            int(b.GetIsConjugated()), int(b.IsInRing())]

def smiles_to_data(smiles, label):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None or mol.GetNumAtoms() == 0: return None
    x = torch.tensor([atom_features(a) for a in mol.GetAtoms()], dtype=torch.float)
    ei, ea = [], []
    for b in mol.GetBonds():
        i, j = b.GetBeginAtomIdx(), b.GetEndAtomIdx(); bf = bond_features(b)
        ei += [[i, j], [j, i]]; ea += [bf, bf]
    if len(ei) == 0:
        edge_index = torch.zeros((2, 0), dtype=torch.long)
        edge_attr  = torch.zeros((0, BOND_DIM), dtype=torch.float)
    else:
        edge_index = torch.tensor(ei, dtype=torch.long).t().contiguous()
        edge_attr  = torch.tensor(ea, dtype=torch.float)
    return Data(x=x, edge_index=edge_index, edge_attr=edge_attr,
                y=torch.tensor([label], dtype=torch.float), smiles=smiles)

def build_dataset(d):
    g = [smiles_to_data(r["canonical"], float(r["label"])) for _, r in d.iterrows()]
    return [x for x in g if x is not None]

class GNN(nn.Module):
    def __init__(self, kind, in_dim=ATOM_DIM, edge_dim=BOND_DIM, hidden=128,
                 n_layers=3, dropout=0.2, heads=4, residual=True, use_edge_attr=True):
        super().__init__()
        self.kind, self.dropout, self.residual = kind, dropout, residual
        self.use_edge_attr = use_edge_attr and kind in ("gin", "gat")
        self.lin_in = nn.Linear(in_dim, hidden)
        self.convs, self.bns = nn.ModuleList(), nn.ModuleList()
        for _ in range(n_layers):
            if kind == "gcn": conv = GCNConv(hidden, hidden)
            elif kind == "gin":
                mlp = nn.Sequential(nn.Linear(hidden, hidden), nn.ReLU(),
                                    nn.Linear(hidden, hidden))
                conv = (GINEConv(mlp, train_eps=True, edge_dim=edge_dim)
                        if self.use_edge_attr else GINConv(mlp, train_eps=True))
            elif kind == "gat":
                conv = GATConv(hidden, hidden // heads, heads=heads,
                               edge_dim=edge_dim if self.use_edge_attr else None)
            else: raise ValueError(kind)
            self.convs.append(conv); self.bns.append(nn.BatchNorm1d(hidden))
        self.head = nn.Sequential(nn.Linear(2*hidden, hidden), nn.ReLU(),
                                  nn.Dropout(dropout), nn.Linear(hidden, 1))
    def forward(self, data):
        x, ei, ea, b = data.x, data.edge_index, data.edge_attr, data.batch
        x = self.lin_in(x)
        for conv, bn in zip(self.convs, self.bns):
            h = conv(x, ei, ea) if self.use_edge_attr else conv(x, ei)
            h = F.relu(bn(h)); h = F.dropout(h, p=self.dropout, training=self.training)
            x = x + h if self.residual else h
        g = torch.cat([global_mean_pool(x, b), global_add_pool(x, b)], dim=1)
        return self.head(g).squeeze(-1)

def build_gnn(kind, hidden, n_layers, dropout, residual=True, heads=4, use_edge_attr=True):
    return GNN(kind, hidden=hidden, n_layers=n_layers, dropout=dropout,
               residual=residual, heads=heads, use_edge_attr=use_edge_attr).to(DEVICE)

class TensorView:
    def __init__(self, x, edge_index, edge_attr, batch):
        self.x, self.edge_index, self.edge_attr, self.batch = x, edge_index, edge_attr, batch

@torch.no_grad()
def evaluate(model, loader):
    model.eval(); ys, ps = [], []
    for d in loader:
        d = d.to(DEVICE)
        ys.append(d.y.view(-1).cpu().numpy())
        ps.append(torch.sigmoid(model(d)).cpu().numpy())
    return np.concatenate(ys), np.concatenate(ps)

def train_gnn(kind, params, seed, tr_set, va_set, pos_weight,
              max_epochs=250, patience=30, use_edge_attr=True, epoch_hook=None):
    """The protocol used for every model in this study."""
    set_seed(seed)
    model = build_gnn(kind, params["hidden"], params["n_layers"], params["dropout"],
                      params.get("residual", True), use_edge_attr=use_edge_attr)
    opt = torch.optim.Adam(model.parameters(), lr=params["lr"],
                           weight_decay=params["weight_decay"])
    crit = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
    tl = DataLoader(tr_set, batch_size=params["batch_size"], shuffle=True, drop_last=True)
    vl = DataLoader(va_set, batch_size=256)
    best, best_state, waited = -1.0, None, 0
    for ep in range(max_epochs):
        model.train()
        for d in tl:
            d = d.to(DEVICE); opt.zero_grad()
            crit(model(d), d.y.view(-1)).backward(); opt.step()
        y, p = evaluate(model, vl)
        auc = roc_auc_score(y, p) if len(np.unique(y)) > 1 else 0.5
        if epoch_hook is not None: epoch_hook(ep, model, auc)
        if auc > best:
            best, waited = auc, 0
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        else:
            waited += 1
            if waited >= patience: break
    model.load_state_dict(best_state); model.eval()
    return model, best

def train_to_fit(kind, params, seed, dataset, n_epochs, dropout=0.0, weight_decay=0.0,
                 lr=None):
    """Fit a target with regularisation and early stopping switched off."""
    set_seed(seed)
    model = build_gnn(kind, params["hidden"], params["n_layers"], dropout,
                      params.get("residual", True))
    opt = torch.optim.Adam(model.parameters(), lr=lr or params["lr"],
                           weight_decay=weight_decay)
    crit = nn.BCEWithLogitsLoss()
    ld = DataLoader(dataset, batch_size=params["batch_size"], shuffle=True, drop_last=True)
    last = float("nan")
    for ep in range(n_epochs):
        model.train(); tot = 0.0
        for d in ld:
            d = d.to(DEVICE); opt.zero_grad()
            loss = crit(model(d), d.y.view(-1)); loss.backward(); opt.step()
            tot += loss.item()
        last = tot / max(len(ld), 1)
    model.eval(); return model, last

Writing /content/ames9.py


In [4]:
import sys, json, glob
sys.path.insert(0, "/content")
import numpy as np, pandas as pd, torch
from ames9 import *
from sklearn.metrics import roc_auc_score as _auc

SEED, N_BOOT = 42, 10000
PRIMARY_SPLIT_SEED = 101
HP_ALL = json.load(open(f"{R}/hpo_gnn.json"))

def cached(name, fn):
    p = f"{R}/{name}"
    if os.path.exists(p):
        print(f"  {name}: loaded from disk"); return pd.read_csv(p)
    out = fn()
    if "note" in out.columns or not len(out):
        print(f"  {name}: stage could not run, not cached"); return out
    out.to_csv(p, index=False); print(f"  {name}: written"); return out

def holm(p):
    p = np.asarray(p, dtype=float)
    order = np.argsort(p); k = len(p); run = 0.0; adj = np.empty(k)
    for r_, i_ in enumerate(order):
        run = max(run, (k - r_)*p[i_]); adj[i_] = min(run, 1.0)
    return adj

def boot_p(draws):
    """Two-sided bootstrap p, bounded away from zero and capped at one.

    The +1 in numerator and denominator keeps a p of exactly zero out of the ledger. The cap
    matters as well: a draw distribution split evenly about zero gives 2*(B/2 + 1)/(B + 1),
    which exceeds one, and a value above one is not a p-value.
    """
    n_le = int((draws <= 0).sum()); n_ge = int((draws >= 0).sum()); B = len(draws)
    return float(min(1.0, 2*min((1+n_le)/(1+B), (1+n_ge)/(1+B))))

df = pd.concat([pd.read_csv(f"{DATADIR}/{f}") for f in ["train.csv","valid.csv","test.csv"]],
               ignore_index=True)[["Drug_ID","smiles","label"]]
df["canonical"] = df["smiles"].apply(
    lambda s: Chem.MolToSmiles(Chem.MolFromSmiles(s)) if Chem.MolFromSmiles(s) else None)
df = df.dropna(subset=["canonical"]).drop_duplicates(
    subset="canonical", keep="first").reset_index(drop=True)
assert len(df) == 7255, f"{len(df)} compounds after curation, expected 7255"

# the manuscript states the designated partition came from the size-ordered rule; this
# reproduces it and checks that claim against the stored file
_, _, df_test = [x.reset_index(drop=True)
                 for x in make_scaffold_split(df, seed=PRIMARY_SPLIT_SEED)]
prim = pd.read_csv(f"{R}/primary_test.csv")
assert set(df_test["canonical"]) == set(prim["canonical"]), (
    "make_scaffold_split does not reproduce the designated test set; the provenance claim "
    "in the manuscript would be wrong")
print(f"designated partition reproduced from the size-ordered splitter: {len(prim)} compounds")

designated partition reproduced from the size-ordered splitter: 726 compounds


In [5]:
TOX = {"aromatic_nitro":"[a][N+](=O)[O-]","aromatic_amine":"[a][NX3;H2,H1;!$(N[!#6]);!$(N=*)]",
 "aromatic_nitroso":"[a][NX2]=[OX1]","alkyl_nitrite":"[CX4][OX2][NX2]=[OX1]",
 "nitrosamine":"[NX3][NX2]=[OX1]","epoxide":"C1OC1","aziridine":"C1NC1",
 "azide":"[NX2]=[NX2+]=[NX1-]","azo_aromatic":"[a][NX2]=[NX2][a]",
 "aliphatic_halide":"[CX4][F,Cl,Br,I]","hetero_hetero_bond":"[N,O;H1,H2][N,O;H1,H2]",
 "polycyclic_aromatic":"c1ccc2ccccc2c1","aromatic_hydroxylamine":"[a][NX3;H1][OX2;H1]",
 "michael_acceptor":"[CX3]=[CX3][CX3]=[OX1]","aromatic_azoxy":"[a][NX2]=[NX3+]([O-])[a]",
 "nitrogen_mustard":"[NX3]([CX4][CX4][F,Cl,Br,I])[CX4][CX4][F,Cl,Br,I]",
 "aromatic_hydrazine":"[a][NX3;H1][NX3;H2]","alkyl_sulfonate":"[CX4][OX2][SX4](=[OX1])(=[OX1])"}
PAT = {k: Chem.MolFromSmarts(v) for k, v in TOX.items()}

def eval_set(frame, labels={1}):
    ev = []
    for smi, lab in zip(frame["canonical"], frame["label"]):
        if lab not in labels: continue
        mol = Chem.MolFromSmiles(smi)
        if mol is None or mol.GetNumAtoms() == 0: continue
        t = set()
        for p in PAT.values():
            for mt in mol.GetSubstructMatches(p): t.update(mt)
        if not t or len(t) == mol.GetNumAtoms(): continue
        ev.append({"smiles": smi, "mol": mol, "tox": t, "n_atoms": mol.GetNumAtoms()})
    return ev, np.array([len(e["tox"])/e["n_atoms"] for e in ev])

EVAL, CHANCE = eval_set(prim)
NEG, _ = eval_set(prim, {0})        # the specificity check scores AUROC, not lift
assert len(EVAL) == 237, f"evaluation set has {len(EVAL)}, expected 237"
assert len(NEG) == 168, f"non-mutagenic set has {len(NEG)}, expected 168"
N_EVAL = len(EVAL)
DEG = [np.array([a.GetDegree() for a in e["mol"].GetAtoms()], dtype=float) for e in EVAL]
NEG_DEG = [np.array([a.GetDegree() for a in e["mol"].GetAtoms()], dtype=float) for e in NEG]

# alert membership as a boolean mask, built once rather than inside every scoring loop
TOXMASK = []
for e in EVAL:
    _m = np.zeros(e["n_atoms"], dtype=bool); _m[list(e["tox"])] = True
    TOXMASK.append(_m)
print(f"mutagenic {len(EVAL)}, non-mutagenic {len(NEG)}")

def per_molecule_auroc(scores, ev=None):
    ev = EVAL if ev is None else ev
    out = np.full(len(ev), np.nan)
    for i, (e, sc) in enumerate(zip(ev, scores)):
        if sc is None or len(sc) != e["n_atoms"]: continue
        y = np.zeros(e["n_atoms"]); y[list(e["tox"])] = 1
        if 0 < y.sum() < len(y): out[i] = _auc(y, np.asarray(sc, dtype=float))
    return out

def common_support(per, ref):
    """Restrict both arms to the molecules on which both are scored.

    Averaging each arm over whatever it happens to have would take the difference between two
    different molecule sets. Every comparison here is paired, so the pairing is enforced once,
    before any resampling.
    """
    ok = ~np.isnan(ref) & ~np.isnan(per).any(axis=0)
    assert ok.sum() >= 0.95*len(ref), \
        f"only {int(ok.sum())} of {len(ref)} molecules are scored on both arms"
    return per[:, ok], ref[ok], ok


def paired_draws(per, ref, n_boot=N_BOOT, seed=SEED):
    """Molecules and models resampled together; the difference taken within each replicate."""
    per, ref, _ = common_support(per, ref)
    rng = np.random.default_rng(seed); n_mod, n_mol = per.shape
    out = np.empty(n_boot)
    for b in range(n_boot):
        mi = rng.integers(0, n_mol, n_mol); si = rng.integers(0, n_mod, n_mod)
        out[b] = per[np.ix_(si, mi)].mean() - ref[mi].mean()
    return out

FILES = {"gin": [f for f in sorted(glob.glob(f"{EXPL}/scores_ig_seed*.npz"))
                 if "_gat_" not in f and "_gcn_" not in f and "no_degree" not in f],
         "gat": sorted(glob.glob(f"{EXPL}/scores_ig_gat_seed*.npz")),
         "gcn": sorted(glob.glob(f"{EXPL}/scores_ig_gcn_seed*.npz"))}
for a, v in FILES.items():
    assert len(v) == 20, f"{a}: {len(v)} score files, expected 20"
EXTRA = {"Gradient x Input": sorted(glob.glob(f"{EXPL}/scores_gradinput_seed*.npz")),
         "Saliency": sorted(glob.glob(f"{EXPL}/scores_saliency_seed*.npz"))}
for k, v in EXTRA.items():
    assert len(v) == 20, f"{k}: {len(v)} score files, expected 20; the arm would enter the "\
                         f"family with a different model count from the others"
    print(f"  {k}: {len(v)} files")
print("stored attribution scores located")

mutagenic 237, non-mutagenic 168
  Gradient x Input: 20 files
  Saliency: 20 files
stored attribution scores located


## 2. Threshold-free comparisons

Five arms form the attribution family: three architectures under Integrated Gradients and two
further gradient methods. The random ranking is a control on the measure and sits outside it.

The random arm needs a resampling of its own. A single random ranking is one draw from the
space of random rankings, and an interval computed on it describes that draw rather than the
class. Twenty independent rankings are used instead, so the arm carries the same two axes as
every other.

In [6]:
def load_scores(path):
    z = np.load(path)
    return [z[f"m{i}"] for i in range(N_EVAL)]

FAMILY_SEED = SEED + 7        # one seed for the whole family, so arms share the resampling

def build_auroc_p():
    d_auc = per_molecule_auroc(DEG)
    rows = []

    rng = np.random.default_rng(SEED + 1)
    rnd_per = np.array([per_molecule_auroc([rng.random(e["n_atoms"]) for e in EVAL])
                        for _ in range(20)])
    dr = paired_draws(rnd_per, d_auc, seed=FAMILY_SEED)
    p_, r_, ok_ = common_support(rnd_per, d_auc)
    rows.append({"arm": "random ranking", "family": "control", "n": 20,
                 "n_eval": int(ok_.sum()), "difference": float(p_.mean() - r_.mean()),
                 "ci_low": float(np.percentile(dr, 2.5)),
                 "ci_high": float(np.percentile(dr, 97.5)), "p_raw": boot_p(dr)})
    print(f"    random (20 rankings): p_raw {rows[-1]['p_raw']:.4f}")

    for arch in ["gin", "gat", "gcn"]:
        per = np.array([per_molecule_auroc(load_scores(f)) for f in FILES[arch]])
        dr = paired_draws(per, d_auc, seed=FAMILY_SEED)
        p_, r_, ok_ = common_support(per, d_auc)
        rows.append({"arm": f"trained {arch.upper()}, IG", "family": "attribution",
                     "n": len(per), "n_eval": int(ok_.sum()),
                     "difference": float(p_.mean() - r_.mean()),
                     "ci_low": float(np.percentile(dr, 2.5)),
                     "ci_high": float(np.percentile(dr, 97.5)), "p_raw": boot_p(dr)})
        print(f"    {arch}: p_raw {rows[-1]['p_raw']:.4f}")

    for lab, fs in EXTRA.items():
        per = np.array([per_molecule_auroc(load_scores(f)) for f in fs])
        dr = paired_draws(per, d_auc, seed=FAMILY_SEED)
        p_, r_, ok_ = common_support(per, d_auc)
        rows.append({"arm": lab, "family": "attribution", "n": len(per), "n_eval": int(ok_.sum()),
                     "difference": float(p_.mean() - r_.mean()),
                     "ci_low": float(np.percentile(dr, 2.5)),
                     "ci_high": float(np.percentile(dr, 97.5)), "p_raw": boot_p(dr)})
        print(f"    {lab}: p_raw {rows[-1]['p_raw']:.4f}")

    out = pd.DataFrame(rows)
    fam = out.family == "attribution"
    assert fam.sum() == 5, f"attribution family has {int(fam.sum())} members, expected 5"
    out.loc[fam, "p_holm"] = holm(out.loc[fam, "p_raw"].values)
    out["resolved"] = np.where(fam, out.p_holm < 0.05, out.p_raw < 0.05)
    return out

aur = cached("table92_auroc_pvalues.csv", build_auroc_p)
print()
print(aur.round(4).to_string(index=False))
print()
print("  Holm applied across the five attribution comparisons. The random arm carries no")
print("  corrected value because it is not in the family; its p is reported uncorrected.")

    random (20 rankings): p_raw 0.0002
    gin: p_raw 0.0036
    gat: p_raw 0.1680
    gcn: p_raw 0.2538
    Gradient x Input: p_raw 0.0766
    Saliency: p_raw 0.0018
  table92_auroc_pvalues.csv: written

             arm      family  n  n_eval  difference  ci_low  ci_high  p_raw  p_holm  resolved
  random ranking     control 20     237     -0.0867 -0.1058  -0.0679 0.0002     NaN      True
 trained GIN, IG attribution 20     237      0.0564  0.0187   0.0926 0.0036  0.0144      True
 trained GAT, IG attribution 20     237      0.0148 -0.0068   0.0365 0.1680  0.3360     False
 trained GCN, IG attribution 20     237      0.0179 -0.0127   0.0482 0.2538  0.3360     False
Gradient x Input attribution 20     237      0.0297 -0.0034   0.0629 0.0766  0.2298     False
        Saliency attribution 20     237      0.0633  0.0241   0.1021 0.0018  0.0090      True

  Holm applied across the five attribution comparisons. The random arm carries no
  corrected value because it is not in the family; its

## 3. The specificity check

The Conclusions rest on this comparison: GIN ranks alert atoms lower where they do not carry
the endpoint. It is reported with an interval and no correction, and it is the strongest
positive claim in the paper. Leaving the strongest claim uncorrected is the asymmetry this
notebook exists to remove, so the three differences of differences are given bootstrap
$p$-values and Holm across the three architectures.

The scores must be recomputed because the comparison uses a signed aggregation, which the
stored files do not carry.

In [7]:
POS_G = [smiles_to_data(e["smiles"], 1.0) for e in EVAL]
NEG_G = [smiles_to_data(e["smiles"], 0.0) for e in NEG]
N_SPEC = 10

def signed_ig(model, graphs, n_steps=32, bs=64):
    was, saved = model.training, [p.requires_grad for p in model.parameters()]
    model.eval()
    for p in model.parameters(): p.requires_grad_(False)
    out = []
    for batch in DataLoader(graphs, batch_size=bs, shuffle=False):
        batch = batch.to(DEVICE); x0 = batch.x.clone()
        total = torch.zeros_like(x0)
        for al in torch.linspace(1.0/n_steps, 1.0, n_steps):
            xa = (x0*al).detach().requires_grad_(True)
            lg = model(TensorView(xa, batch.edge_index, batch.edge_attr, batch.batch))
            total += torch.autograd.grad(lg.sum(), xa)[0]
        attr = (x0*(total/n_steps)).sum(1).detach().cpu().numpy()
        b = batch.batch.cpu().numpy()
        for gi in range(batch.num_graphs): out.append(attr[b == gi])
    for p, f in zip(model.parameters(), saved): p.requires_grad_(f)
    model.train(was)
    return out

def load_models(arch, n):
    hp = HP_ALL[arch]; out = []
    for f in sorted(glob.glob(f"{MODELS}/{arch}_attr_seed*.pt"))[:n]:
        m = build_gnn(arch, hp["hidden"], hp["n_layers"], hp["dropout"],
                      hp.get("residual", True))
        m.load_state_dict(torch.load(f, map_location=DEVICE,
                                     weights_only=False)["state_dict"])
        m.eval(); out.append(m)
    return out

def build_dod_p():
    d_pos = per_molecule_auroc(DEG); d_neg = per_molecule_auroc(NEG_DEG, ev=NEG)
    rows = []
    for arch in ["gin", "gat", "gcn"]:
        mdls = load_models(arch, N_SPEC)
        assert len(mdls) == N_SPEC, f"{arch}: {len(mdls)} weights found, expected {N_SPEC}"
        P, N = [], []
        for idx in range(len(mdls)):
            m = mdls[idx]
            P.append(per_molecule_auroc(signed_ig(m, POS_G)))
            N.append(per_molecule_auroc(signed_ig(m, NEG_G), ev=NEG))
            mdls[idx] = None          # the list held the only other reference
            del m
            if torch.cuda.is_available(): torch.cuda.empty_cache()
        # each set is reduced to the molecules both its arms are scored on, as elsewhere
        P, p_ref, ok_p = common_support(np.array(P), d_pos)
        N, n_ref, ok_n = common_support(np.array(N), d_neg)
        rng = np.random.default_rng(SEED + 3)
        bb = np.empty(N_BOOT)
        for b in range(N_BOOT):
            si = rng.integers(0, P.shape[0], P.shape[0])       # models shared across sets
            pi = rng.integers(0, P.shape[1], P.shape[1])
            ni = rng.integers(0, N.shape[1], N.shape[1])
            dp = P[np.ix_(si, pi)].mean() - p_ref[pi].mean()
            dn = N[np.ix_(si, ni)].mean() - n_ref[ni].mean()
            bb[b] = dn - dp
        dp0 = float(P.mean() - p_ref.mean())
        dn0 = float(N.mean() - n_ref.mean())
        lo, hi = np.percentile(bb, [2.5, 97.5])
        p_raw = boot_p(bb)
        rows.append({"architecture": arch.upper(), "n_models": N_SPEC,
                     "n_mutagenic": int(ok_p.sum()), "n_non_mutagenic": int(ok_n.sum()),
                     "diff_mutagenic": dp0, "diff_non_mutagenic": dn0,
                     "dod": dn0 - dp0, "ci_low": lo, "ci_high": hi, "p_raw": p_raw})
        print(f"    {arch}: dod {dn0-dp0:+.4f} [{lo:+.4f}, {hi:+.4f}]  p_raw {p_raw:.4f}")
    out = pd.DataFrame(rows)
    out["p_holm"] = holm(out.p_raw.values)
    out["resolved"] = out.p_holm < 0.05
    return out

dod = cached("table96_dod_pvalues.csv", build_dod_p)
print()
print(dod.round(4).to_string(index=False))
print()
print("  The pre-registered direction is negative. Holm across the three architectures.")

    gin: dod -0.0498 [-0.0904, -0.0114]  p_raw 0.0122
    gat: dod -0.0238 [-0.0650, +0.0177]  p_raw 0.2634
    gcn: dod +0.0004 [-0.0417, +0.0415]  p_raw 0.9747
  table96_dod_pvalues.csv: written

architecture  n_models  n_mutagenic  n_non_mutagenic  diff_mutagenic  diff_non_mutagenic     dod  ci_low  ci_high  p_raw  p_holm  resolved
         GIN        10          237              168          0.0890              0.0393 -0.0498 -0.0904  -0.0114 0.0122  0.0366      True
         GAT        10          237              168         -0.0317             -0.0554 -0.0238 -0.0650   0.0177 0.2634  0.5267     False
         GCN        10          237              168          0.0310              0.0314  0.0004 -0.0417   0.0415 0.9747  0.9747     False

  The pre-registered direction is negative. Holm across the three architectures.


## 4. The retained-fraction sweep

The sweep is used to say that no architecture exceeds the baseline at the reference fraction.
That is a claim of absence, and a correction can only make a claim of absence harder to
overturn. Leaving those twelve uncorrected is therefore the conservative reading for the
statement drawn from them.

The $p$-values are computed anyway, because the manuscript should be able to show that the
uncorrected reading was a choice and not an omission, and because the sweep also carries a
claim of presence at the smallest fraction, where the published table puts GIN and GAT
above the baseline. That claim is corrected; the claim of absence is not.

The sweep is recomputed here rather than read from the published table, because a percentile
interval does not determine a two-sided $p$. A recomputation invites a discrepancy, so the
result at the reference fraction is compared with the published value and the reading for a
disagreement is fixed in advance.

In [8]:
K_GRID = [0.10, 0.20, 0.30, 0.40, 0.50]
N_TIE = 200          # the same budget on both arms, so the comparison carries no asymmetry

# the published sweep at the reference fraction, for the agreement check below
PUBLISHED_030 = {"GIN": 0.052, "GAT": -0.011, "GCN": -0.061}


def mean_overlap(scores, k_frac, n_tie=N_TIE, base=0, ev=None, masks=None):
    """Share of the retained set inside an alert, averaged over tie-breaks, per molecule.

    Ties are broken exactly, by a random secondary key under lexsort. Adding a small jitter
    to the scores instead would be faster to write and wrong twice over: it reorders values
    that differ by less than the jitter, and where the scores are large enough that the
    jitter falls below their floating-point spacing it does not separate ties at all, so the
    average over tie-breaks silently collapses to one deterministic draw.

    The tie-breaks are vectorised inside each molecule rather than looped outside it, which
    is about three times faster and lets both arms of a comparison carry the same budget.
    Seeding per molecule keeps common random numbers between arms: the first m rows of an
    n-row block are the block that would have been drawn for m, so a molecule receives the
    same keys whichever arm is being scored.
    """
    ev = EVAL if ev is None else ev
    masks = TOXMASK if masks is None else masks
    out = np.full(len(ev), np.nan)
    for i, (e, sc, tm) in enumerate(zip(ev, scores, masks)):
        if sc is None or len(sc) != e["n_atoms"]: continue
        n = e["n_atoms"]; k = max(1, int(round(k_frac*n)))
        v = -np.asarray(sc, dtype=float)
        keys = np.random.default_rng([base, i]).random((n_tie, n))
        order = np.lexsort((keys, np.broadcast_to(v, (n_tie, n))), axis=-1)
        out[i] = tm[order[:, :k]].sum(1).mean()/k
    return out


def paired_lift_draws(model_ov, deg_ov, chance, n_boot=N_BOOT, seed=SEED):
    """(model lift - degree lift), molecules and models resampled together.

    Both arms are restricted to the same molecules before this is called, so the difference
    is taken on one set rather than on two that a missing cell could pull apart.

    The tie-break axis is not resampled: both arms are tie-break averages computed before
    this function is called, so the interval carries molecules and models but not the
    tie-breaking. The manuscript reports that spread separately.
    """
    rng = np.random.default_rng(seed); n_mod, n_mol = model_ov.shape
    out = np.empty(n_boot)
    for b in range(n_boot):
        mi = rng.integers(0, n_mol, n_mol); si = rng.integers(0, n_mod, n_mod)
        c = chance[mi].mean()
        out[b] = (model_ov[np.ix_(si, mi)].mean() - deg_ov[mi].mean())/c
    return out


def sweep_one_fraction(kf):
    """One retained fraction, cached on its own so a dropped session costs one fraction."""
    base = 40000 + int(kf*100)
    d_ov = mean_overlap(DEG, kf, base=base)
    rows = []
    for arch in ["gin", "gat", "gcn"]:
        ovs = np.array([mean_overlap(load_scores(f), kf, base=base) for f in FILES[arch]])
        m_o, d_o, ok = common_support(ovs, d_ov)
        ch = CHANCE[ok]
        d_l = float(d_o.mean()/ch.mean())
        m_l = float(m_o.mean()/ch.mean())
        dr = paired_lift_draws(m_o, d_o, ch, seed=FAMILY_SEED)
        lo, hi = np.percentile(dr, [2.5, 97.5])
        rows.append({"k_fraction": kf, "architecture": arch.upper(), "n_eval": int(ok.sum()),
                     "degree_lift": d_l, "model_lift": m_l,
                     "difference": m_l - d_l, "ci_low": lo, "ci_high": hi,
                     "p_raw": boot_p(dr), "n_tiebreaks": N_TIE,
                     "excludes_zero": bool(lo > 0 or hi < 0)})
    return pd.DataFrame(rows)


def build_sweep_p():
    """The sweep recomputed from the stored attribution scores, with bootstrap p-values.

    A percentile interval does not determine a two-sided p, so the sweep is recomputed from
    the scores rather than approximated from the published intervals.
    """
    parts = []
    for kf in K_GRID:
        part = cached(f"table93_sweep_k{int(kf*100):02d}.csv",
                      lambda kf=kf: sweep_one_fraction(kf))
        # a cache written under a different tie-break budget is not this analysis
        assert "n_tiebreaks" in part.columns and (part.n_tiebreaks == N_TIE).all(), (
            f"k = {kf}: cached under a different tie-break budget or an older column "
            f"layout; delete table93_sweep_k{int(kf*100):02d}.csv and rerun")
        parts.append(part)
        print(f"    k = {kf:.2f} done")
    out = pd.concat(parts, ignore_index=True)
    top = out.k_fraction == min(K_GRID)
    out.loc[top, "p_holm_top_fraction"] = holm(out.loc[top, "p_raw"].values)
    out["family"] = "k sweep"
    return out


sweep = cached("table93_sweep_pvalues.csv", build_sweep_p)
# build_sweep_p checks each per-fraction cache, but it does not run at all when the combined
# file is already on disk, so the combined file is checked here for the same thing
assert {"n_tiebreaks", "p_raw", "p_holm_top_fraction"} <= set(sweep.columns) \
    and (sweep.n_tiebreaks == N_TIE).all() and len(sweep) == 3*len(K_GRID), (
    "table93_sweep_pvalues.csv was written under a different tie-break budget or column "
    "layout; delete it and the table93_sweep_k*.csv files and rerun")
print()
print(sweep.round(4).to_string(index=False))

# does the recomputation reproduce the published table at the reference fraction
print()
print("  recomputation against the published sweep at k = 0.30")
r30 = sweep[sweep.k_fraction == 0.30]
assert len(r30) == 3, "the reference fraction is missing from the sweep"
worst = 0.0
for _, r in r30.iterrows():
    pub = PUBLISHED_030[r.architecture]; d = abs(r.difference - pub)
    worst = max(worst, d)
    print(f"    {r.architecture}: {r.difference:+.4f} vs published {pub:+.4f}   "
          f"|delta| = {d:.4f}")
print(f"  largest disagreement {worst:.4f}; the pre-registered threshold is 0.01")
print("  agrees, the published table stands and the p-values are appended to it"
      if worst <= 0.01 else
      "  disagrees, the recomputed table replaces tab_kfraction and the difference is "
      "reported as an effect of the tie-break budget")

top = sweep[sweep.p_holm_top_fraction.notna()]
n_rest = len(sweep) - len(top)
print()
print(f"  at k = {min(K_GRID):.2f}, {int((top.p_holm_top_fraction < 0.05).sum())} of "
      f"{len(top)} architectures survive Holm across that subfamily")
print(f"  the remaining {n_rest} cells carry uncorrected bootstrap p-values, for the reason")
print("  given above: the claim drawn from them is one of absence")

  table93_sweep_k10.csv: written
    k = 0.10 done
  table93_sweep_k20.csv: written
    k = 0.20 done
  table93_sweep_k30.csv: written
    k = 0.30 done
  table93_sweep_k40.csv: written
    k = 0.40 done
  table93_sweep_k50.csv: written
    k = 0.50 done
  table93_sweep_pvalues.csv: written

 k_fraction architecture  n_eval  degree_lift  model_lift  difference  ci_low  ci_high  p_raw  n_tiebreaks  excludes_zero  p_holm_top_fraction  family
        0.1          GIN     237       1.2009      1.4305      0.2296  0.1169   0.3505 0.0002          200           True               0.0006 k sweep
        0.1          GAT     237       1.2009      1.3376      0.1367  0.0623   0.2158 0.0002          200           True               0.0006 k sweep
        0.1          GCN     237       1.2009      1.2561      0.0552 -0.0375   0.1469 0.2382          200          False               0.2382 k sweep
        0.2          GIN     237       1.2196      1.3467      0.1271  0.0393   0.2179 0.0046          

## 5. One decision rule, tabulated

Every comparison the manuscript makes against the degree ranking, with its family, the
correction applied and the threshold. Test counts are read from the tables rather than typed,
so the row cannot drift from what the table contains.

In [9]:
def count_rows(path, col=None):
    if not os.path.exists(path): return None
    d = pd.read_csv(path)
    return int(len(d) if col is None else d[col].notna().sum())

def build_rules():
    n_auroc = int((aur.family == "attribution").sum())
    n_dod = int(len(dod))
    n_top = int((sweep.k_fraction == sweep.k_fraction.min()).sum())
    n_rest = int(len(sweep)) - n_top
    n_paired = count_rows(f"{R}/table70_paired_architecture.csv")
    n_reg = count_rows(f"{R}/table76_regime_paired.csv")
    n_rep = count_rows(f"{R}/table87_headline_by_partition.csv")
    n_repreg = count_rows(f"{R}/table88_regimes_by_partition.csv")
    n_panel = count_rows(f"{R}/table62_regime_intervals.csv")
    rows = [
        ("Architectures, threshold-free", "tab:auroc", n_auroc, "Holm",
         "0.05 on the corrected $p$"),
        ("Random ranking against degree", "tab:auroc", 1,
         "none, a control on the measure", "0.05"),
        ("Architectures, lift scale", "tab:paired", n_paired, "Holm",
         "0.05 on the corrected $p$"),
        ("Regime decomposition", "tab:regime_trained", n_reg, "Holm",
         "0.05 on the corrected $p$"),
        ("Specificity, difference of differences", "tab:dod", n_dod, "Holm",
         "0.05 on the corrected $p$"),
        ("Retained-fraction sweep, smallest fraction", "tab:kfraction", n_top, "Holm",
         "0.05 on the corrected $p$"),
        ("Retained-fraction sweep, larger fractions", "tab:kfraction", n_rest,
         "none, the claim drawn from them is one of absence",
         "interval excluding zero"),
        ("Replication, three partitions", "tab:replication", n_rep,
         "none, three pre-specified partitions", "interval excluding zero"),
        ("Replication, regimes", "tab:replication_regimes", n_repreg,
         "none; the corresponding family was corrected on the designated partition",
         "interval excluding zero"),
        ("Structural ranking panel", "tab:regimes", n_panel,
         "none, a screen that selected the baseline", "interval excluding chance"),
    ]
    out = pd.DataFrame(rows, columns=["Comparison","Table","Tests","Correction","Threshold"])
    missing = out[out.Tests.isna()]
    if len(missing):
        # cached() looks for a note column or an empty frame; a NaN count is neither, so
        # without this the incomplete table is written and never recomputed
        print("  source table not found for:", list(missing.Comparison))
        out["note"] = "incomplete: a source table was not found"
    return out

rules = cached("table94_decision_rules.csv", build_rules)
print(rules.to_string(index=False))

# the counts above are read from the tables; the families were fixed before the run. A
# disagreement means one of the two is wrong, and it should be visible rather than absorbed.
PREREG = {"Architectures, threshold-free": 5,
          "Specificity, difference of differences": 3,
          "Retained-fraction sweep, smallest fraction": 3,
          "Retained-fraction sweep, larger fractions": 12,
          "Structural ranking panel": 33}
print()
bad = [(c, PREREG[c], r) for c, r in zip(rules.Comparison, rules.Tests)
       if c in PREREG and pd.notna(r) and int(r) != PREREG[c]]
if bad:
    for c, want, got in bad:
        print(f"  MISMATCH {c}: pre-registered {want}, table has {int(got)}")
    print("  resolve this before quoting either number in the manuscript")
else:
    print("  every derived count matches the family sizes fixed before the run")

  source table not found for: ['Replication, regimes']
  table94_decision_rules.csv: stage could not run, not cached
                                Comparison                   Table  Tests                                                               Correction                 Threshold                                     note
             Architectures, threshold-free               tab:auroc    5.0                                                                     Holm 0.05 on the corrected $p$ incomplete: a source table was not found
             Random ranking against degree               tab:auroc    1.0                                           none, a control on the measure                      0.05 incomplete: a source table was not found
                 Architectures, lift scale              tab:paired    3.0                                                                     Holm 0.05 on the corrected $p$ incomplete: a source table was not found
                      Regim

## 6. Summary

In [10]:
print("threshold-free comparisons")
print(aur[["arm","family","n","difference","ci_low","ci_high","p_raw","p_holm","resolved"]]
      .round(4).to_string(index=False))
res = aur[(aur.family == "attribution") & aur.resolved]
n_fam = int((aur.family == "attribution").sum())
print(f"  {len(res)} of {n_fam} survive Holm: {list(res.arm)}")

print("\nspecificity")
print(dod[["architecture","dod","ci_low","ci_high","p_raw","p_holm","resolved"]]
      .round(4).to_string(index=False))
res2 = dod[dod.resolved]
print(f"  {len(res2)} of {len(dod)} survive Holm: {list(res2.architecture)}")

top = sweep[sweep.p_holm_top_fraction.notna()]
print(f"\nsweep at k = {top.k_fraction.iloc[0]:.2f}: "
      f"{int((top.p_holm_top_fraction < 0.05).sum())} of {len(top)} survive Holm")

num = {}
for _, r in aur.iterrows():
    k = r["arm"].lower().replace(" ", "_").replace(",", "")
    num[f"aurocp_{k}_raw"] = float(r.p_raw)
    if pd.notna(r.get("p_holm")): num[f"aurocp_{k}_holm"] = float(r.p_holm)
for _, r in dod.iterrows():
    num[f"dodp_{r.architecture}_raw"] = float(r.p_raw)
    num[f"dodp_{r.architecture}_holm"] = float(r.p_holm)
for _, r in sweep.iterrows():
    kk = int(round(r.k_fraction*100))
    num[f"sweepp_{r.architecture}_k{kk:02d}_raw"] = float(r.p_raw)
    if pd.notna(r.get("p_holm_top_fraction")):
        num[f"sweepp_{r.architecture}_k{kk:02d}_holm"] = float(r.p_holm_top_fraction)
pd.DataFrame(sorted(num.items()), columns=["quantity","value"]).to_csv(
    f"{R}/table95_part18_ledger.csv", index=False)
print(f"\n{len(num)} quantities written for merging into the manuscript ledger")
for f in ["table92_auroc_pvalues.csv","table93_sweep_pvalues.csv",
          "table94_decision_rules.csv","table96_dod_pvalues.csv",
          "table95_part18_ledger.csv"]:
    print("  -", f, "yes" if os.path.exists(f"{R}/{f}") else "MISSING")
import shutil
from google.colab import files
shutil.make_archive("/content/part18_results", "zip", R)
files.download("/content/part18_results.zip")

threshold-free comparisons
             arm      family  n  difference  ci_low  ci_high  p_raw  p_holm  resolved
  random ranking     control 20     -0.0867 -0.1058  -0.0679 0.0002     NaN      True
 trained GIN, IG attribution 20      0.0564  0.0187   0.0926 0.0036  0.0144      True
 trained GAT, IG attribution 20      0.0148 -0.0068   0.0365 0.1680  0.3360     False
 trained GCN, IG attribution 20      0.0179 -0.0127   0.0482 0.2538  0.3360     False
Gradient x Input attribution 20      0.0297 -0.0034   0.0629 0.0766  0.2298     False
        Saliency attribution 20      0.0633  0.0241   0.1021 0.0018  0.0090      True
  2 of 5 survive Holm: ['trained GIN, IG', 'Saliency']

specificity
architecture     dod  ci_low  ci_high  p_raw  p_holm  resolved
         GIN -0.0498 -0.0904  -0.0114 0.0122  0.0366      True
         GAT -0.0238 -0.0650   0.0177 0.2634  0.5267     False
         GCN  0.0004 -0.0417   0.0415 0.9747  0.9747     False
  1 of 3 survive Holm: ['GIN']

sweep at k = 0.10: 

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>